# 2.2 — Detección de anomalías / outliers

**Aspecto del rúbrica:** *Anomalías (outliers), usando al menos dos técnicas*.

Se aplican **mínimo dos** técnicas de detección y se comparan resultados:
eventos climáticos extremos (olas de calor), picos de contaminación y picos
epidémicos pueden ser outliers *legítimos* — distinguirlos de errores de datos
es parte del análisis.

## Técnicas a aplicar (elegir ≥2, se sugieren 3)

1. **Regla IQR / Z-score** (univariado, interpretable):
   - `Q1 - 1.5*IQR`, `Q3 + 1.5*IQR`; Z-score `|z| > 3`.
2. **Isolation Forest** (`sklearn.ensemble`): detección multivariada no
   paramétrica; requiere estandarizar (`StandardScaler`).
3. **Local Outlier Factor (LOF)** (`sklearn.neighbors`): densidad local;
   útil para eventos que son normales en su contexto pero raros globalmente.

## Entradas esperadas

| Ruta | Contenido |
|------|-----------|
| `data/processed/panel_hermosillo.parquet` | Panel integrado |

## Salidas

- Figuras en `reports/figures/` (boxplots, series con outliers marcados, scatter Isolation Forest).
- Tabla comparativa de outliers detectados por técnica.
- Justificación de qué outliers se conservan (eventos reales) vs. se tratan.

## Configuración e imports

In [ ]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent.parent  # notebooks/<seccion>/ -> raiz del proyecto
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

# TODO: cargar el panel integrado
# panel = pd.read_parquet("data/processed/panel_hermosillo.parquet", parse_dates=["fecha"])

# TODO: definir variables numéricas candidatas a análisis de outliers
# (excluir IDs, claves INEGI, porcentajes acotados)
# VARS_OUTLIER = ["temperatura_maxima", "pm25", "pm10", "o3", "casos_ira", ...]


## Técnica 1 — Regla IQR / Z-score (univariado)

In [ ]:
# TODO: función auxiliar para marcar outliers por IQR y por Z-score
def outliers_iqr(s: pd.Series, k: float = 1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)

def outliers_zscore(s: pd.Series, threshold: float = 3.0):
    z = (s - s.mean()) / s.std()
    return z.abs() > threshold

# TODO: aplicar a cada variable y resumir conteos
# for var in VARS_OUTLIER:
#     n_iqr = outliers_iqr(panel[var]).sum()
#     n_z = outliers_zscore(panel[var]).sum()
#     print(f"{var}: IQR={n_iqr}, Z-score={n_z}")


## Visualización univariada

In [ ]:
# TODO: boxplots por variable (señalan outliers visualmente)
# panel[VARS_OUTLIER].boxplot(figsize=(12, 6), rot=45)
# plt.tight_layout()
# plt.savefig("reports/figures/outliers_boxplot.png", dpi=150)
# plt.show()

# TODO: series temporales con outliers IQR marcados
# for var in VARS_OUTLIER:
#     mask = outliers_iqr(panel[var])
#     ax = panel.loc[~mask, "fecha"].astype(str)  # placeholder
#     plt.figure(figsize=(12, 3))
#     plt.plot(panel["fecha"], panel[var], label=var)
#     plt.scatter(panel.loc[mask, "fecha"], panel.loc[mask, var],
#                 color="red", label="IQR outlier", zorder=5)
#     plt.legend(); plt.tight_layout()
#     plt.savefig(f"reports/figures/outliers_iqr_{var}.png", dpi=150)
#     plt.show()


## Técnica 2 — Isolation Forest (multivariado)

In [ ]:
# TODO: estandarizar y ajustar Isolation Forest
# X = panel[VARS_OUTLIER].select_dtypes(include=[np.number]).dropna()
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

# iso = IsolationForest(n_estimators=200, contamination="auto", random_state=42)
# iso_labels = iso.fit_predict(X_scaled)          # -1 = outlier, 1 = normal
# iso_scores = -iso.score_samples(X_scaled)       # mayor = más anómalo

# panel_out = panel.loc[X.index].copy()
# panel_out["iso_outlier"] = iso_labels == -1
# panel_out["iso_score"] = iso_scores
# print("Outliers Isolation Forest:", panel_out["iso_outlier"].sum())


## Técnica 3 — Local Outlier Factor (densidad local)

In [ ]:
# TODO: LOF sobre los mismos datos estandarizados
# lof = LocalOutlierFactor(n_neighbors=20, novelty=False)
# lof_labels = lof.fit_predict(X_scaled)          # -1 = outlier, 1 = normal
# lof_scores = -lof.negative_outlier_factor_      # mayor = más anómalo

# panel_out["lof_outlier"] = lof_labels == -1
# print("Outliers LOF:", panel_out["lof_outlier"].sum())


## Comparación de técnicas

In [ ]:
# TODO: tabla comparativa y acuerdo entre técnicas
# comparacion = pd.DataFrame({
#     "fecha": panel_out["fecha"],
#     "iso": panel_out["iso_outlier"],
#     "lof": panel_out["lof_outlier"],
# })
# print(comparacion.groupby(["iso", "lof"]).size())

# TODO: scatter de scores (Isolation Forest vs LOF) coloreado por fecha
# plt.figure(figsize=(7, 6))
# plt.scatter(panel_out["iso_score"], panel_out["lof_score"],
#             c=pd.to_datetime(panel_out["fecha"]).astype(int), cmap="viridis", s=10)
# plt.xlabel("Isolation Forest score"); plt.ylabel("LOF score")
# plt.colorbar(label="fecha")
# plt.tight_layout()
# plt.savefig("reports/figures/outliers_score_scatter.png", dpi=150)
# plt.show()


## Interpretación (rellenar con hallazgos)

In [ ]:
# TODO: documentar qué outliers son eventos reales (olas de calor,
# picos de contaminación, brotes epidemiológicos) vs. errores de datos,
# y qué decisión se toma con cada uno (conservar / corregir / excluir).


## Consulta con el equipo (checkpoint 1)

Registrar aquí lo que se presentó al equipo sobre los outliers detectados y su tratamiento y qué se decidió
(ver `docs/docs/protocolo_eda.md`, sección 5.3).

| Fecha | Participantes | Hallazgo o pregunta presentada | Decisión |
|-------|---------------|-------------------------------|----------|
| _TODO_ | | | |